# 01 · Prueba de conexión a PostgreSQL y Redis

Verifica que los servicios del stack son accesibles desde JupyterLab. Los hosts (`postgres`, `redis`) son los nombres de servicio de Docker Compose y las credenciales se leen de variables de entorno (definidas en `.env`).

Los drivers (`psycopg2`, `redis`) ya vienen instalados en la imagen de Jupyter del proyecto, no hace falta `pip install`.

## PostgreSQL

In [2]:
%pip install psycopg2-binary

   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.9 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.9 MB 886.1 kB/s eta 0:00:03
   ---------- ----------------------------- 0.8/2.9 MB 1.1 MB/s eta 0:00:02
   ------------------ --------------------- 1.3/2.9 MB 1.3 MB/s eta 0:00:02
   --------------------- ------------------ 1.6/2.9 MB 1.3 MB/s eta 0:00:02
   ------------------------- -------------- 1.8/2.9 MB 1.3 MB/s eta 0:00:01
   ----------------------------- ---------- 2.1/2.9 MB 1.4 MB/s eta 0:00:01
   ------------------------------------ --- 2.6/2.9 MB 1.4 MB/s eta 0:00:01
   ---------------------------------------- 2.9/2.9 MB 1.5 MB/s  0:00:02
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import psycopg2

conn = psycopg2.connect(
    host=os.environ["POSTGRES_HOST"],
    port=5432,
    user=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
    dbname=os.environ["POSTGRES_DB"],
)

with conn.cursor() as cur:
    cur.execute("SELECT version();")
    print(cur.fetchone()[0].split(",")[0])

    for tabla in ["clientes", "productos", "pedidos", "detalle_pedidos", "marketing.contactos"]:
        cur.execute(f"SELECT COUNT(*) FROM {tabla};")
        print(f"{tabla:<22}{cur.fetchone()[0]:>8} filas")

    cur.execute("SELECT nombre, ciudad FROM clientes ORDER BY id LIMIT 5;")
    for fila in cur.fetchall():
        print(fila)

conn.close()

KeyError: 'POSTGRES_HOST'

Resultado esperado: 50 clientes, 12 productos, 100 pedidos y 200 líneas de pedido. `marketing.contactos` tendrá **0 filas hasta que ejecutes el ETL** y 45.211 después:

```bash
docker compose run --rm etl
```

## Redis (con contraseña)

In [2]:
import redis

host = os.environ["REDIS_HOST"]

r = redis.Redis(host=host, port=6379, password=os.environ["REDIS_PASSWORD"], decode_responses=True)
print("PING con contraseña:", r.ping())

r.set("pipeline_status", "ejecutando", ex=60)  # caduca a los 60 s
print("pipeline_status =", r.get("pipeline_status"))

PING con contraseña: True
pipeline_status = ejecutando


Redis exige contraseña: una conexión anónima debe ser rechazada.

In [3]:
anonimo = redis.Redis(host=host, port=6379, socket_timeout=2)
try:
    anonimo.ping()
    print("⚠️ Redis acepta conexiones sin contraseña")
except redis.AuthenticationError:
    print("✔ Redis rechaza las conexiones sin contraseña")

✔ Redis rechaza las conexiones sin contraseña
